In [1]:
import requests
from bs4 import BeautifulSoup, Comment

import pandas as pd
import re
import time

Spotrac

In [ ]:
def get_table_headings(table): 
    table_headings = table.thead.find_all("th")
    table_heading_list = [] 
    for heading in table_headings: 
        heading_text = re.sub(r"\s\(\d+\)", "", heading.text.replace('\n','').strip())
        table_heading_list.append(re.sub(r"(?<!^)(?=[A-Z])", "_", heading_text.replace(' ','')).lower())
    return table_heading_list

def parse_payroll_tables(response_text, team, year): 
    # Parse the html code 
    soup = BeautifulSoup(response_text,"html.parser")

    # Pull the table names from the page 
    table_names = soup.select('div[class*="table-header"]')
    table_name_list = []
    for name in table_names: 
        table_name_list.append(name.h2.text.replace('\n','').strip())
    
    #print(table_name_list)

    # Pull all the tables 
    tables = soup.find_all("table")

    # List to hold all of the dataframes 
    payroll_df_list = [] 
    # Headings for each payroll detail table 
    payroll_df_headings = get_table_headings(tables[0])
    # Going through each table 
    num_tables = len(tables)
    for i in range(0,num_tables): 
        table = tables[i]
        # There is a table with summary which is different from the others and contains ranks instead of players 
        if (table['id'] == "table") & (i != num_tables - 1): 
            table_body = table.tbody.find_all("tr")
            rows = [] 
            for row in table_body: 
                row_data = row.find_all("td")
                columns = []
                if len(row_data) == 3: 
                    for col in row_data: 
                        columns.append(col.text.replace('\n', '').strip())
                    rows.append(columns)
            payroll_ranks = pd.DataFrame(rows, columns=["player_type", "amount", "rank"])
            payroll_ranks["team"] = team
            payroll_ranks["year"] = year 
        else: 
            table_body = table.tbody.find_all("tr")
            rows = [] 
            for row in table_body: 
                row_data = row.find_all("td")
                columns = []
                for col in row_data: 
                    if col.find("a"): 
                        columns.append(col.a.text)
                    elif col.find("span"):  
                        columns.append(col.span.text.replace('\n','').strip())
                    else: 
                        columns.append(col.text.replace('\n', '').strip())
                rows.append(columns)
            df = pd.DataFrame(rows, columns=payroll_df_headings)
            df["roster"] = table_name_list[i]
            payroll_df_list.append(df)
    
    payroll_data = pd.concat(payroll_df_list, ignore_index=True)
    payroll_data["team"] = team
    payroll_data["year"] = year

    return (payroll_ranks, payroll_data)

In [18]:
teams = [
    "arizona-diamondbacks", 
    "athletics", 
    "atlanta-braves", 
    "baltimore-orioles", 
    "boston-red-sox", 
    "chicago-cubs", 
    "chicago-white-sox", 
    "cincinnati-reds", 
    "cleveland-guardians", 
    "colorado-rockies", 
    "detroit-tigers", 
    "houston-astros", 
    "kansas-city-royals", 
    "los-angeles-angels", 
    "los-angeles-dodgers", 
    "miami-marlins", 
    "milwaukee-brewers", 
    "minnesota-twins", 
    "new-york-mets", 
    "new-york-yankees", 
    "philadelphia-phillies", 
    "pittsburgh-pirates", 
    "san-diego-padres", 
    "san-francisco-giants", 
    "seattle-mariners", 
    "st-louis-cardinals", 
    "tampa-bay-rays", 
    "texas-rangers", 
    "toronto-blue-jays", 
    "washington-nationals"
]
years = list(range(2015, 2025))

In [ ]:
rank_df_list = []
payroll_df_list = []

for year in years: 
    for team in teams: 
        url = f"https://www.spotrac.com/mlb/{team}/payroll/_/year/{year}"
        print(url)
        try: 
            response = requests.get(url)
        except Exception as e:
            print(response)
            break 
        payroll_ranks, payroll_data = parse_payroll_tables(response.text, team, year)
        rank_df_list.append(payroll_ranks)
        payroll_df_list.append(payroll_data)

ranks = pd.concat(rank_df_list, ignore_index=True)
payroll = pd.concat(payroll_df_list, ignore_index=True)

In [ ]:
ranks.to_csv("payroll_ranks_from_2015.csv")
payroll.to_csv("payroll_data_from_2015.csv")

Baseball Reference - 40 man

In [ ]:
response = requests.get("https://www.baseball-reference.com/about/team_IDs.shtml")

In [ ]:
soup = BeautifulSoup(response.text, "html.parser")
table_body = soup.table.find_all("tr")

rows = []
for row in table_body:
    columns = []
    row_data = row.find_all("td")
    for col in row_data: 
        columns.append(col.text)
    rows.append(columns)

df = pd.DataFrame(rows)
df.columns = df.iloc[0]  # Set the first row as column names
df = df[1:]  # Remove the first row from the data

# Reset the index
df.reset_index(drop=True, inplace=True)

current_teams = df[df["Last Year"] == "Present"]

br_team_apprev = current_teams["Team ID"]

In [ ]:
def get_40_man_table_headings(table): 
    table_heading_list = []
    for heading in table.thead.find_all("th"): 
        table_heading_list.append(heading.text)
    return table_heading_list

def parse_40_man_table(table, team, year): 
    table_body = table.tbody.find_all("tr")
    rows = []
    for row in table_body:
        row_name = row.th.text
        columns = [row_name]
        row_data = row.find_all("td")
        for col in row_data: 
            columns.append(col.text)
        rows.append(columns)

    forty_man = pd.DataFrame(rows, columns=get_40_man_table_headings(table))
    forty_man["team"] = team
    forty_man["year"] = year
    return forty_man

In [ ]:
url = f"https://www.baseball-reference.com/teams/WSN/2024-roster.shtml#all_appearances"
print(url)
response = requests.get(url)

response.encoding = "utf-8"
soup = BeautifulSoup(response.text,"html.parser")

In [ ]:
tables = soup.find_all("table")
roster_detail_table = None
for table in tables:
    caption = table.find("caption")
    if caption and caption.text == "Full-Season Roster & Games by Position":
        roster_detail_table = table

In [ ]:
comments = soup.find_all(string=lambda text: isinstance(text, Comment))
for comment in comments:
    if '<div class="table_container tabbed current" id="div_appearances">' in comment:
        #roster_detail_table = BeautifulSoup(comment, "html.parser")
        table = comment 

In [ ]:
forty_man_df_list = []

for year in years: 
    for team in br_team_apprev: 
        url = f"https://www.baseball-reference.com/teams/{team}/{year}-roster.shtml#all_appearances"
        print(url)
        response = requests.get(url)
        
        if response.status_code != 200: 
            print(response.status_code)
            break 
        
        response.encoding = "utf-8"
        soup = BeautifulSoup(response.text,"html.parser")

        tables = soup.find_all("table")
        roster_detail_table = None
        for table in tables:
            caption = table.find("caption")
            if caption and caption.text == "Full-Season Roster & Games by Position":
                roster_detail_table = table
                break
        forty_man = parse_40_man_table(roster_detail_table, team, year)
        forty_man_df_list.append(forty_man)
        time.sleep(5)
    time.sleep(30)

forty_man = pd.concat(forty_man_df_list, ignore_index=True)

In [ ]:
forty_man.to_csv("bb_ref_forty_man_roster_from_2015.csv")

Baseball Reference - Standings

In [ ]:
def extract_standings_table(html_code):
    expanded_standings = None
    comments = html_code.find_all(string=lambda text: isinstance(text, Comment))
    for comment in comments:
        if '<div class="table_container" id="div_expanded_standings_overall">' in comment:
            expanded_standings = BeautifulSoup(comment, "html.parser")
    return expanded_standings
    
def get_standings_table_headings(table): 
    table_heading_list = []
    for heading in table.thead.find_all("th"): 
        table_heading_list.append(heading.text)
    return table_heading_list

def parse_standings_table(table, year): 
    table_body = table.tbody.find_all("tr")
    rows = []
    for row in table_body:
        row_name = row.th.text
        columns = [row_name]
        row_data = row.find_all("td")
        for col in row_data: 
            columns.append(col.text)
        rows.append(columns)

    standings = pd.DataFrame(rows, columns=get_standings_table_headings(table))
    standings["year"] = year
    return standings

In [ ]:
standings_df_list = []

years = years = list(range(2012, 2025))
for year in years: 
    url = f"https://www.baseball-reference.com/leagues/majors/{year}-standings.shtml#all_expanded_standings_overall"
    print(url)
    response = requests.get(url)
    
    if response.status_code != 200: 
        print(response.status_code)
        break 
    
    soup = BeautifulSoup(response.text,"html.parser")

    standings_table = extract_standings_table(soup)
    
    standings = parse_standings_table(standings_table , year)
    standings_df_list.append(standings)
    time.sleep(5)


standings = pd.concat(standings_df_list, ignore_index=True)